In [1]:
# Imports and global constants

import os
import random
import shutil
import subprocess
from pathlib import Path

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import yaml
import pandas as pd
import torch
import cv2

# Adjust these to your setup:
BASE     = Path('/home/admin/Documents/AI_Eng/Week_06')
YOLO_DIR = BASE / 'yolov5'

# Device for inference/training
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'


In [2]:
# Convert CSV annotations → YOLO txt format

def convert_annotations(csv_path: Path, labels_dir: Path, class_map={'Graffiti': 0}):
    """
    Convert CSV bbox annotations to YOLO txt files.
    """
    df = pd.read_csv(csv_path)
    labels_dir.mkdir(parents=True, exist_ok=True)

    for img_name, group in df.groupby('filename'):
        w, h = group[['width','height']].iloc[0]
        lines = []
        for _, row in group.iterrows():
            cid = class_map[row['class']]
            x_c = ((row.xmin + row.xmax) / 2) / w
            y_c = ((row.ymin + row.ymax) / 2) / h
            bw  = (row.xmax - row.xmin) / w
            bh  = (row.ymax - row.ymin) / h
            lines.append(f"{cid} {x_c:.6f} {y_c:.6f} {bw:.6f} {bh:.6f}")

        out_file = labels_dir / f"{Path(img_name).stem}.txt"
        out_file.write_text("\n".join(lines))

convert_annotations(BASE/'train_labels.csv', BASE/'labels'/'train')
convert_annotations(BASE/'test_labels.csv',  BASE/'labels'/'test')


In [3]:
# Prepare dataset folder structure + graffiti.yaml

def prepare_dataset(src_img_train: Path,
                    src_img_test:  Path,
                    lbl_train_dir:  Path,
                    lbl_test_dir:   Path,
                    dst:            Path,
                    n_train=400,
                    n_val=40,
                    seed=None):
    """
    Sample and copy images + labels into YOLOv5 structure, then write graffiti.yaml.
    """
    random.seed(seed)
    dst_img_tr = dst/'images'/'train'
    dst_img_va = dst/'images'/'val'
    dst_lbl_tr = dst/'labels'/'train'
    dst_lbl_va = dst/'labels'/'val'
    for p in (dst_img_tr, dst_img_va, dst_lbl_tr, dst_lbl_va):
        p.mkdir(parents=True, exist_ok=True)

    all_train = list(src_img_train.glob('*.jpg'))
    all_test  = list(src_img_test.glob('*.jpg'))
    train_sel = random.sample(all_train, n_train)
    val_sel   = random.sample(all_test,  n_val)

    # copy images and corresponding .txt labels
    for imgs, dst_img, src_lbl, dst_lbl in [
        (train_sel, dst_img_tr, lbl_train_dir, dst_lbl_tr),
        (val_sel,   dst_img_va, lbl_test_dir,  dst_lbl_va)
    ]:
        for img in imgs:
            shutil.copy(img, dst_img/img.name)
            txt = src_lbl / f"{img.stem}.txt"
            if txt.exists():
                shutil.copy(txt, dst_lbl/f"{img.stem}.txt")

    # Write dataset YAML
    yaml_dict = {
        'path': str(dst),
        'train': 'images/train',
        'val':   'images/val',
        'nc':    1,
        'names': ['Graffiti']
    }
    with open(dst/'graffiti.yaml','w') as f:
        yaml.dump(yaml_dict, f)


In [4]:
# Helper functions for training & evaluation

def train_yolo(data_yaml: str,
               weights:    str,
               exp_name:   str,
               epochs=30,
               img_size=640,
               batch=16,
               cache=True):
    """
    Runs yolov5/train.py inside YOLO_DIR.
    """
    cmd = [
        'python', 'train.py',
        '--img',   str(img_size),
        '--batch', str(batch),
        '--epochs',str(epochs),
        '--data',  data_yaml,
        '--weights', weights,
        '--name',  exp_name
    ] + (['--cache'] if cache else [])

    subprocess.run(cmd,
                   cwd=str(YOLO_DIR),
                   check=True)

def load_ground_truth(txt_path: Path, img_w: int, img_h: int):
    boxes = []
    if not txt_path.exists():
        return boxes
    for line in txt_path.read_text().splitlines():
        _, x_c, y_c, w, h = map(float, line.split())
        x1 = (x_c - w/2) * img_w
        y1 = (y_c - h/2) * img_h
        x2 = (x_c + w/2) * img_w
        y2 = (y_c + h/2) * img_h
        boxes.append([x1,y1,x2,y2])
    return boxes

def compute_iou(boxA, boxB):
    xA, yA = max(boxA[0], boxB[0]), max(boxA[1], boxB[1])
    xB, yB = min(boxA[2], boxB[2]), min(boxA[3], boxB[3])
    inter = max(0, xB-xA)*max(0, yB-yA)
    areaA = (boxA[2]-boxA[0])*(boxA[3]-boxA[1])
    areaB = (boxB[2]-boxB[0])*(boxB[3]-boxB[1])
    return inter / (areaA + areaB - inter + 1e-6)

def evaluate_model(model, img_paths, lbl_dir, save_csv):
    rows = []
    for img_path in img_paths:
        img = cv2.imread(str(img_path))
        h,w = img.shape[:2]
        preds = model(img).xyxy[0].cpu().numpy()
        gts   = load_ground_truth(lbl_dir/f"{img_path.stem}.txt", w, h)

        if len(preds)==0:
            rows.append([img_path.name, 0.0, 0.0])
        else:
            best = preds[preds[:,4].argmax()]
            x1,y1,x2,y2,conf,_ = best
            ious = [compute_iou([x1,y1,x2,y2], g) for g in gts]
            rows.append([img_path.name, float(conf), max(ious) if ious else 0.0])

    df = pd.DataFrame(rows, columns=['image_name','confidence','IoU'])
    df.to_csv(save_csv, index=False)
    return (df['IoU']>=0.9).mean()


In [5]:
# Enhanced training function that caches and explicitly saves each best.pt

import shutil

def train_yolo_and_save(data_yaml: str,
                        weights:    str,
                        exp_name:   str,
                        epochs=30,
                        img_size=640,
                        batch=16,
                        cache=True):
    """
    Runs yolov5/train.py with caching, then copies best.pt to yolov5/models/exp_name.pt
    """
    cmd = [
        'python', 'train.py',
        '--img',    str(img_size),
        '--batch',  str(batch),
        '--epochs', str(epochs),
        '--data',   data_yaml,
        '--weights',weights,
        '--name',   exp_name
    ] + (['--cache'] if cache else [])

    # 1) launch training
    subprocess.run(cmd,
                   cwd=str(YOLO_DIR),
                   check=True)

    # 2) locate the best weights
    src = YOLO_DIR / 'runs' / 'train' / exp_name / 'weights' / 'best.pt'
    
    # 3) copy to a central models folder
    models_dir = YOLO_DIR / 'models'
    models_dir.mkdir(parents=True, exist_ok=True)
    dst = models_dir / f"{exp_name}.pt"
    shutil.copy(src, dst)
    print(f"Saved best.pt : {dst}")

    return str(dst)

In [ ]:
# %% Cell X: full iterative training+evaluation

data_yaml     = str(BASE/'dataset'/'graffiti.yaml')
prev_weights  = 'yolov5s.pt'   # initial pretrained
max_iters     = 10

for i in range(1, max_iters+1):
    # 1) re-sample into dataset/images/train & val
    prepare_dataset(
        src_img_train=BASE/'images'/'train',
        src_img_test= BASE/'images'/'test',
        lbl_train_dir=BASE/'labels'/'train',
        lbl_test_dir= BASE/'labels'/'test',
        dst=BASE/'dataset',
        n_train=400,
        n_val=40,
        seed=None               # different sample each time
    )
    
    # 2) train and save best.pt
    exp_name     = f'graffiti_exp{i}'
    best_weights = train_yolo_and_save(data_yaml, prev_weights, exp_name, epochs=30)
    
    # 3) evaluate on this iteration’s 40
    val_imgs     = list((BASE/'dataset'/'images'/'val').glob('*.jpg'))
    rate         = evaluate_model(
                       torch.hub.load('ultralytics/yolov5','custom',
                                      path=best_weights, force_reload=False).to(DEVICE),
                       val_imgs,
                       BASE/'dataset'/'labels'/'val',
                       save_csv=f'Iteration_Results/iteration{i}_results.csv'
                   )
    print(f"Iteration {i}: {rate*100:.1f}% ≥ 0.9 IoU")
    
    # 4) break if success
    if rate >= 0.8:
        print("Reached 80% of images ≥ 90% IoU; stopping.")
        break
    
    # prepare for next iteration
    prev_weights = best_weights


2025-05-05 13:40:43.281438: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1746416443.291417  384605 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1746416443.294327  384605 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1746416443.302686  384605 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1746416443.302698  384605 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1746416443.302699  384605 computation_placer.cc:177] computation placer alr

In [9]:
# %% Cell 10: Organise CSV + pick two “good” samples per iteration

import pandas as pd
import shutil
from pathlib import Path
import cv2

def collect_iteration_outputs(iteration,
                              model,
                              dataset_base,
                              results_folder='outputs'):
    """
    - Moves iteration{i}_results.csv → outputs/iteration{i}/
    - Picks top-2 by IoU, runs inference & writes their rendered images
      into outputs/iteration{i}/samples/
    """
    # paths
    csv_src    = Path(f'Iteration_Results/iteration{iteration}_results.csv')
    out_iter   = Path(results_folder)/f'iteration{iteration}'
    samples_dir= out_iter/'samples'
    val_imgs   = dataset_base/'images'/'val'
    lbl_dir    = dataset_base/'labels'/'val'
    
    # make directories
    samples_dir.mkdir(parents=True, exist_ok=True)
    
    # 1) move CSV
    csv_dst = out_iter/csv_src.name
    shutil.move(str(csv_src), str(csv_dst))
    
    # 2) pick top 2 by IoU
    df = pd.read_csv(csv_dst)
    top2 = df.sort_values('IoU', ascending=False).head(10)['image_name'].tolist()
    
    # 3) for each top image: run model & save rendered image
    for img_name in top2:
        img_path = val_imgs/img_name
        img      = cv2.imread(str(img_path))
        res      = model(img)             # yolo inference
        rendered = res.render()[0]        # BGR array
        
        out_path = samples_dir/img_name
        cv2.imwrite(str(out_path), rendered)
        print(f"Saved sample : {out_path}")

# Usage example, immediately after your loop:
# -------------------------------------------
from pathlib import Path
for i in range(1, max_iters+1):    # achieved_iter is the last iteration you ran
    weights = YOLO_DIR/'models'/f'graffiti_exp{i}.pt'
    model   = torch.hub.load('ultralytics/yolov5','custom',
                             path=str(weights), force_reload=False).to(DEVICE)
    collect_iteration_outputs(i,
                               model,
                               BASE/'dataset',
                               results_folder='outputs')


Using cache found in /home/admin/.cache/torch/hub/ultralytics_yolov5_master
YOLOv5 🚀 2025-4-29 Python-3.10.16 torch-2.7.0+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 7836MiB)

Fusing layers... 
Model summary: 157 layers, 7012822 parameters, 0 gradients, 15.8 GFLOPs
Adding AutoShape... 


FileNotFoundError: [Errno 2] No such file or directory: 'Iteration_Results/iteration1_results.csv'

In [ ]:
#  Visualize detections on a set of images inline

import matplotlib.pyplot as plt
from IPython.display import display
from pathlib import Path

def visualize_batch(model, image_paths, out_folder="vis_outputs"):
    """
    Runs model on each image, saves rendered results to out_folder,
    then displays them inline.
    """
    out_folder = Path(out_folder)
    out_folder.mkdir(exist_ok=True)
    
    for img_path in image_paths:
        img = cv2.imread(str(img_path))
        res = model(img)
        rendered = res.render()[0]               # numpy array with boxes drawn
        save_to = out_folder / img_path.name
        cv2.imwrite(str(save_to), rendered)
        
        # convert BGR→RGB for display
        rgb = cv2.cvtColor(rendered, cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(6,6))
        plt.imshow(rgb)
        plt.axis('off')
        display(plt.gcf())
        plt.close()

# Example usage:
final_model = YOLO_DIR/'models'/'graffiti_exp10.pt'
viz_model   = torch.hub.load('ultralytics/yolov5','custom', path=str(final_model), force_reload=False).to(DEVICE)
imgs = list((BASE/'dataset'/'images'/'val').glob('*.jpg'))
visualize_batch(viz_model, imgs[:40])    # show first 40


In [16]:
# Single‐video inference

def process_video(video_path, weights_path, output_path, class_names=['Graffiti']):
    """
    Runs YOLOv5 on each frame of video_path, writes annotated video to output_path.
    """
    # load model
    model = torch.hub.load('ultralytics/yolov5','custom', path=weights_path, force_reload=False).to(DEVICE)

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open {video_path}")

    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps    = cap.get(cv2.CAP_PROP_FPS) or 30.0
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out    = cv2.VideoWriter(str(output_path), fourcc, fps, (width, height))

    print(f"Processing {video_path} → {output_path}")
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        results = model(frame)
        frame   = results.render()[0]
        out.write(frame)

    cap.release()
    out.release()
    print("Done.")

input_vid  = BASE/'videos'/'1.mp4'
output_vid = BASE/'videos'/'1_detected.mp4'
process_video(input_vid, str(YOLO_DIR/'models'/'graffiti_exp10.pt'), output_vid)

input_vid  = BASE/'videos'/'2.mp4'
output_vid = BASE/'videos'/'2_detected.mp4'
process_video(input_vid, str(YOLO_DIR/'models'/'graffiti_exp10.pt'), output_vid)

input_vid  = BASE/'videos'/'3.mp4'
output_vid = BASE/'videos'/'3_detected.mp4'
process_video(input_vid, str(YOLO_DIR/'models'/'graffiti_exp10.pt'), output_vid)


input_vid  = BASE/'videos'/'4.mp4'
output_vid = BASE/'videos'/'4_detected.mp4'
process_video(input_vid, str(YOLO_DIR/'models'/'graffiti_exp10.pt'), output_vid)

input_vid  = BASE/'videos'/'5.mp4'
output_vid = BASE/'videos'/'5_detected.mp4'
process_video(input_vid, str(YOLO_DIR/'models'/'graffiti_exp10.pt'), output_vid)

Using cache found in /home/admin/.cache/torch/hub/ultralytics_yolov5_master
YOLOv5 🚀 2025-4-29 Python-3.10.16 torch-2.7.0+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 7836MiB)

Fusing layers... 
Model summary: 157 layers, 7012822 parameters, 0 gradients, 15.8 GFLOPs
Adding AutoShape... 


Processing /home/admin/Documents/AI_Eng/Week_06/videos/4.mp4 → /home/admin/Documents/AI_Eng/Week_06/videos/4_detected.mp4
Done.


Using cache found in /home/admin/.cache/torch/hub/ultralytics_yolov5_master
YOLOv5 🚀 2025-4-29 Python-3.10.16 torch-2.7.0+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 7836MiB)

Fusing layers... 
Model summary: 157 layers, 7012822 parameters, 0 gradients, 15.8 GFLOPs
Adding AutoShape... 


Processing /home/admin/Documents/AI_Eng/Week_06/videos/5.mp4 → /home/admin/Documents/AI_Eng/Week_06/videos/5_detected.mp4
Done.
